# Duck Harness — Complete Unit Test Run

This GPU-enabled Kaggle notebook prepares the repository's locked Python 3.12.12 environments and runs the default unit-test suites for both `ARC3-Inference` and `tufa-arc-agi-framework`. JUnit XML reports are written to `/kaggle/working/test-results`.

Enable **Internet** in the notebook settings when no source dataset is attached. If a dataset containing both project directories is attached, the notebook uses that snapshot instead of cloning GitHub. Repository-configured slow/integration tests remain excluded; all default unit tests run.

## 1. Configuration and helpers

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import sysconfig
import xml.etree.ElementTree as ET
from pathlib import Path

REPOSITORY_URL = os.environ.get(
    "DUCK_HARNESS_REPOSITORY",
    "https://github.com/arcainion/duck-harness.git",
)
REPOSITORY_BRANCH = os.environ.get("DUCK_HARNESS_BRANCH", "main")
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
CHECKOUT_DIR = WORKING_DIR / "duck-harness-tests"
RESULTS_DIR = WORKING_DIR / "test-results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def run(command, *, cwd=None, check=True):
    rendered = " ".join(str(part) for part in command)
    print(f"\n$ {rendered}", flush=True)
    return subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        check=check,
    )


## 2. Locate or clone the source

In [ ]:
def find_attached_checkout():
    if not KAGGLE_INPUT.is_dir():
        return None
    for lock_file in KAGGLE_INPUT.rglob("ARC3-Inference/uv.lock"):
        candidate = lock_file.parent.parent
        if (candidate / "tufa-arc-agi-framework" / "uv.lock").is_file():
            return candidate
    return None


def copy_project(source, destination):
    shutil.copytree(
        source,
        destination,
        ignore=shutil.ignore_patterns(
            ".git", ".venv", "__pycache__", ".pytest_cache", ".ruff_cache", "build"
        ),
    )


if CHECKOUT_DIR.exists():
    shutil.rmtree(CHECKOUT_DIR)

attached_checkout = find_attached_checkout()
if attached_checkout is not None:
    print(f"Using attached source snapshot: {attached_checkout}")
    CHECKOUT_DIR.mkdir(parents=True)
    copy_project(attached_checkout / "ARC3-Inference", CHECKOUT_DIR / "ARC3-Inference")
    copy_project(
        attached_checkout / "tufa-arc-agi-framework",
        CHECKOUT_DIR / "tufa-arc-agi-framework",
    )
else:
    print("No attached source snapshot found; cloning GitHub.")
    run(
        [
            "git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH,
            REPOSITORY_URL, CHECKOUT_DIR,
        ]
    )

INFERENCE_DIR = CHECKOUT_DIR / "ARC3-Inference"
FRAMEWORK_DIR = CHECKOUT_DIR / "tufa-arc-agi-framework"
for project_dir in (INFERENCE_DIR, FRAMEWORK_DIR):
    if not (project_dir / "pyproject.toml").is_file():
        raise RuntimeError(f"Missing project source: {project_dir}")
print(f"Checkout ready at {CHECKOUT_DIR}")


## 3. Install locked test environments

Each project gets its own `.venv`. `uv` installs the exact Python version required by both lockfiles.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
UV = shutil.which("uv") or str(Path(sysconfig.get_path("scripts")) / "uv")
if not Path(UV).is_file():
    raise RuntimeError("uv was installed but its executable is not on PATH.")

run(
    [UV, "sync", "--locked", "--extra", "dev", "--python", "3.12.12"],
    cwd=INFERENCE_DIR,
)
run(
    [UV, "sync", "--locked", "--all-groups", "--python", "3.12.12"],
    cwd=FRAMEWORK_DIR,
)


## 4. Run every default unit-test suite

Both suites run even if the first fails, so the notebook produces a complete failure report. The final cell fails when either suite fails.

In [ ]:
suites = [
    (
        "ARC3-Inference",
        INFERENCE_DIR,
        [UV, "run", "--locked", "--extra", "dev", "pytest", "-q"],
    ),
    (
        "tufa-arc-agi-framework",
        FRAMEWORK_DIR,
        [UV, "run", "--locked", "--all-groups", "pytest", "-q"],
    ),
]
suite_results = []
for suite_name, project_dir, command in suites:
    report_path = RESULTS_DIR / f"{suite_name}-junit.xml"
    completed = run(
        [*command, f"--junitxml={report_path}"],
        cwd=project_dir,
        check=False,
    )
    suite_results.append(
        {
            "suite": suite_name,
            "returncode": completed.returncode,
            "report": str(report_path),
        }
    )

(RESULTS_DIR / "summary.json").write_text(
    json.dumps(suite_results, indent=2) + "\n", encoding="utf-8"
)


## 5. Test summary

In [ ]:
def junit_counts(report_path):
    root = ET.parse(report_path).getroot()
    suites = [root] if root.tag == "testsuite" else list(root.findall("testsuite"))
    return {
        key: sum(int(suite.attrib.get(key, 0)) for suite in suites)
        for key in ("tests", "failures", "errors", "skipped")
    }


failed_suites = []
for result in suite_results:
    report = Path(result["report"])
    counts = junit_counts(report) if report.is_file() else {}
    status = "PASS" if result["returncode"] == 0 else "FAIL"
    print(f"{status}: {result['suite']} {counts}")
    if result["returncode"] != 0:
        failed_suites.append(result["suite"])

print(f"JUnit reports: {RESULTS_DIR}")
if failed_suites:
    raise RuntimeError(f"Unit-test failures: {', '.join(failed_suites)}")
print("All Duck Harness unit tests passed.")
